In [5]:
# Load test dataset with the same random seed for data splitting
from pathlib import Path

from transformers import AutoModelForTokenClassification, AutoTokenizer
from spesia_research.datasets import ClinicalRecordsDataset

model_id = r'experiments\exp_1_mecla_mmbert\best_model'
dataset_path = Path('datasets/breast_cancer_dataset')
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(model_id)
test_dataset = ClinicalRecordsDataset(dataset_path, split='test', tokenizer=tokenizer)

Loading all Records: 100%|██████████| 1/1 [00:02<00:00,  2.17s/it]


In [6]:
train_dataset = ClinicalRecordsDataset(dataset_path, split='train', tokenizer=tokenizer)

Loading all Records: 100%|██████████| 1/1 [00:01<00:00,  1.98s/it]


In [11]:
train_dataset.pos_weight

tensor([17.5789, 35.0204,  1.8653,  2.8537,  4.7492,  7.5680, 15.8095,  9.6325,
         2.0223,  6.2041,  2.5657,  1.3012], dtype=torch.float64)

In [138]:
train_dataset.label2id

{'BRCA_NEGATIVO': 0,
 'BRCA_POSITIVO': 1,
 'CIRURGIA': 2,
 'HER2_NEGATIVO': 3,
 'HER2_POSITIVO': 4,
 'POS_MENOPAUSA': 5,
 'PRE_MENOPAUSA': 6,
 'RE_NEGATIVO': 7,
 'RE_POSITIVO': 8,
 'RP_NEGATIVO': 9,
 'RP_POSITIVO': 10,
 'TIPO_HISTOPATOLOGICO': 11}

In [137]:
train_dataset.get_label_count()

TIPO_HISTOPATOLOGICO    767
CIRURGIA                616
RE_POSITIVO             584
RP_POSITIVO             495
HER2_NEGATIVO           458
HER2_POSITIVO           307
RP_NEGATIVO             245
POS_MENOPAUSA           206
RE_NEGATIVO             166
PRE_MENOPAUSA           105
BRCA_NEGATIVO            95
BRCA_POSITIVO            49
dtype: int64

In [19]:
import torch

input = test_dataset[0]
input_ids = torch.tensor(input['input_ids']).unsqueeze(0)
attention_mask = torch.tensor(input['attention_mask']).unsqueeze(0).unsqueeze(-1)
labels = input['labels'].unsqueeze(0)

print(f'Shapes: \n{input_ids.shape=}\n{attention_mask.shape=}\n{labels.shape=}')

Shapes: 
input_ids.shape=torch.Size([1, 490])
attention_mask.shape=torch.Size([1, 490, 1])
labels.shape=torch.Size([1, 490, 12])


In [11]:
mutually_exclusive_classes = [
    ['HER2_POSITIVO', 'HER2_NEGATIVO'],
    ['RE_POSITIVO', 'RE_NEGATIVO'],
    ['RP_POSITIVO', 'RP_NEGATIVO'],
  ]

exclusive_groups = [
                [model.config.label2id[label] for label in group]
                for group in mutually_exclusive_classes
            ]

In [13]:
# Expect three softmax groups: 
exclusive_groups

[[4, 3], [8, 7], [10, 9]]

In [16]:
# Compute logits
logits = model(input_ids, attention_mask=attention_mask).logits

print(f'Shape: {logits.shape=}')

Shape: logits.shape=torch.Size([1, 490, 12])


In [17]:
# Create exclusive groups max
exclusive_idx = sorted({i for g in exclusive_groups for i in g})
exclusive_mask = torch.zeros(
    logits.size(-1), dtype=torch.bool, device=logits.device
)
exclusive_mask[exclusive_idx] = True
exclusive_mask

tensor([False, False, False,  True,  True, False, False,  True,  True,  True,
         True, False])

In [20]:
# Perform BCE on non-exclusive groups
loss_fct = torch.nn.BCEWithLogitsLoss(
    reduction="none", pos_weight=test_dataset.pos_weight
)
bce_full = loss_fct(logits, labels.float())  # [B,L,C]
bce_non_excl = bce_full[..., ~exclusive_mask]  # [B,L,C_non_excl]
loss_bce = (bce_non_excl * attention_mask).sum() / attention_mask.sum().clamp(min=1)

In [23]:
bce_non_excl.shape

torch.Size([1, 490, 6])

In [25]:
attention_mask.shape

torch.Size([1, 490, 1])

In [28]:
# grouped softmax on exclusive groups
B, L, C = logits.shape
device = logits.device
attn2 = torch.tensor(input["attention_mask"]).to(logits.dtype)  # [B,L]
denom = attn2.sum().clamp(min=1)

# print shapes
print(f'Shapes: \n{logits.shape=}\n{attn2.shape=}\n{denom.shape=}')

Shapes: 
logits.shape=torch.Size([1, 490, 12])
attn2.shape=torch.Size([490])
denom.shape=torch.Size([])


In [38]:
mecla_amplification_factor = 1.0

loss_ce_sum = logits.new_zeros(())
for g in exclusive_groups:
    print('label pair indices:', g)
    group_logits = torch.cat(
        [logits.new_zeros((B, L, 1)), logits[..., g]], dim=-1
    )  # [B,L,1+|g|]: NONE=0
    print(f'group_logits shape: {group_logits.shape=}')

    y = labels[..., g].float()  # [B,L,|g|]
    print(f'y shape: {y.shape=}')
    active = y > 0.5
    n_active = active.sum(dim=-1)  # [B,L]
    print(f'n_active shape: {n_active.shape=} {n_active}')

    target = torch.zeros((B, L), dtype=torch.long, device=device)  # NONE=0
    # If exactly one active, set target to its 1-based index
    one_active = n_active == 1
    if one_active.any():
        idx = active[one_active].long().argmax(dim=-1)  # [N]
        target[one_active] = idx + 1

    # If >1 active (contradiction), ignore
    target = target.masked_fill(n_active > 1, -100)

    ce = torch.nn.functional.cross_entropy(
        group_logits.view(-1, group_logits.size(-1)),
        target.view(-1),
        reduction="none",
        ignore_index=-100,
    ).view(B, L)

    loss_ce_sum = loss_ce_sum + (ce * attn2).sum() / denom

# loss = non-exclusive BCE + MECLA-amplified CE on grouped softmax
loss = loss_bce + mecla_amplification_factor * loss_ce_sum

label pair indices: [4, 3]
group_logits shape: group_logits.shape=torch.Size([1, 490, 3])
y shape: y.shape=torch.Size([1, 490, 2])
n_active shape: n_active.shape=torch.Size([1, 490]) tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       

In [53]:
ce = torch.nn.functional.cross_entropy(
        group_logits[..., 1:].view(-1, group_logits[..., 1:].size(-1)),
        target.view(-1),
        reduction="none",
        ignore_index=-100,
    ).view(B, L)

loss_ce_sum = loss_ce_sum + (ce * attn2).sum() / denom
loss_ce_sum

tensor(0.7358, grad_fn=<AddBackward0>)

In [54]:
ce = torch.nn.functional.cross_entropy(
        group_logits.view(-1, group_logits.size(-1)),
        target.view(-1),
        reduction="none",
        ignore_index=-100,
    ).view(B, L)

loss_ce_sum = loss_ce_sum + (ce * attn2).sum() / denom
loss_ce_sum

tensor(0.7360, grad_fn=<AddBackward0>)

In [55]:
logits

tensor([[[-10.4738, -11.0859, -10.6186,  ...,  -9.5319, -10.0436,  -6.9888],
         [-11.6645, -13.5683, -12.1848,  ...,  -9.6200, -10.0736,  -7.1461],
         [-10.7288, -13.0842, -12.6347,  ...,  -9.6890, -11.2535,  -6.4657],
         ...,
         [ -8.1604, -11.9782,  -9.8968,  ..., -10.2146, -11.1702, -10.1741],
         [ -8.5389, -11.9065, -10.5104,  ..., -10.1874, -11.8564, -10.4079],
         [ -8.4362, -12.0134,  -9.9183,  ...,  -9.9374, -11.4107,  -9.5221]]],
       grad_fn=<ViewBackward0>)

In [ ]:
preds = torch.zeros_like(logits)

# preds for mutually exclusive groups
for g in exclusive_groups:
    group_logits = torch.cat(
        [logits.new_zeros((B, L, 1)), logits[..., g]], dim=-1
    )
    # Softmax in the log space for numerical stability
    group_preds = (
        group_logits.exp()/
        torch.logsumexp(group_logits, dim=-1).view(B, L, 1).exp()
        ).argmax(dim=-1)
    # Update preds
    for g_idx, label_idx in enumerate(g):
        preds[..., label_idx] = (group_preds == g_idx+1).float()

# preds for non-exclusive labels
best_thresholds = [v["threshold"] for k, v in model.config.thresholds.items()]
non_exclusive_logits = logits[..., ~exclusive_mask]
non_exclusive_probs = 1 / (1 + torch.exp(-non_exclusive_logits))
non_exclusive_thresholds = torch.tensor(best_thresholds)[~exclusive_mask]
preds[..., ~exclusive_mask] = (non_exclusive_probs > non_exclusive_thresholds).float()

In [115]:
preds = torch.zeros_like(logits)
probs = torch.zeros_like(logits)

# preds for mutually exclusive groups
for g in exclusive_groups:
    group_logits = torch.cat(
        [logits.new_zeros((B, L, 1)), logits[..., g]], dim=-1
    )
    # Softmax in the log space for numerical stability
    group_probs = (
        group_logits.exp()/
        torch.logsumexp(group_logits, dim=-1).view(B, L, 1).exp()
        )
    group_preds = group_probs.argmax(dim=-1)
    # Update preds
    for g_idx, label_idx in enumerate(g):
        preds[..., label_idx] = (group_preds == g_idx+1).float()
    # Update probs
    probs[..., g] = group_probs[..., 1:]

# preds for non-exclusive labels
best_thresholds = [v["threshold"] for k, v in model.config.thresholds.items()]
non_exclusive_logits = logits[..., ~exclusive_mask]
non_exclusive_probs = 1 / (1 + torch.exp(-non_exclusive_logits))
non_exclusive_thresholds = torch.tensor(best_thresholds)[~exclusive_mask]
preds[..., ~exclusive_mask] = (non_exclusive_probs > non_exclusive_thresholds).float()
probs[..., ~exclusive_mask] = non_exclusive_probs

In [133]:
torch.where(~exclusive_mask)[0].tolist()

[0, 1, 2, 5, 6, 11]

In [135]:
torch.full((12,), 0.5)

tensor([0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000,
        0.5000, 0.5000, 0.5000])

In [116]:
probs

tensor([[[2.8265e-05, 1.5327e-05, 2.4456e-05,  ..., 7.2492e-05,
          4.3458e-05, 9.2128e-04],
         [8.5932e-06, 1.2805e-06, 5.1073e-06,  ..., 6.6382e-05,
          4.2175e-05, 7.8728e-04],
         [2.1905e-05, 2.0777e-06, 3.2571e-06,  ..., 6.1959e-05,
          1.2961e-05, 1.5535e-03],
         ...,
         [2.8566e-04, 6.2796e-06, 5.0334e-05,  ..., 3.6630e-05,
          1.4087e-05, 3.8146e-05],
         [1.9566e-04, 6.7463e-06, 2.7250e-05,  ..., 3.7640e-05,
          7.0928e-06, 3.0191e-05],
         [2.1683e-04, 6.0623e-06, 4.9263e-05,  ..., 4.8331e-05,
          1.1076e-05, 7.3210e-05]]], grad_fn=<IndexPutBackward0>)

In [117]:
labels

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]])

In [126]:
import numpy as np
from typing import Literal
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import average_precision_score
import numpy as np
from transformers.trainer_utils import EvalPrediction
from spesia_research.data_models import ThresholdMap

include_per_label_thresholds = True

# Flatten batch and sequence for metrics
probs_flat = probs.reshape(-1, probs.shape[-1])
labels_flat = labels.reshape(-1, labels.shape[-1])

metrics = {}
with torch.no_grad():
    if probs.shape[-1] > 1:
        # Multilabel case — compute a threshold PER LABEL
        thresholds = np.linspace(0.01, 0.99, 99)
        n_labels = probs_flat.shape[-1]
        best_thresholds = torch.full((n_labels,), 0.5)

        for k in range(n_labels):
            y_true = labels_flat[:, k]
            p = probs_flat[:, k]
            best_f1_k = -1.0
            best_t_k = 0.5
            # Skip labels that are all one class to avoid degenerate optimization
            # (we still keep default 0.5)
            if (y_true.sum() == 0) or (y_true.sum() == y_true.shape[0]):
                best_thresholds[k] = best_t_k
                continue
            for t in thresholds:
                y_pred_k = (p > t)
                f1_k = f1_score(y_true, y_pred_k, average="binary", zero_division=0)
                if f1_k > best_f1_k:
                    best_f1_k = f1_k
                    best_t_k = t
            best_thresholds[k] = best_t_k

        # Use per-label thresholds for final predictions
        preds_flat = (probs_flat > best_thresholds)

        if include_per_label_thresholds:
            metrics["best_thresholds"] = best_thresholds

        metrics["best_thresholds_mean"] = best_thresholds.mean().item()

        metrics["macro_precision"] = precision_score(
            labels_flat, preds_flat, average="macro", zero_division=0
        )
        metrics["macro_recall"] = recall_score(
            labels_flat, preds_flat, average="macro", zero_division=0
        )
        metrics["macro_f1"] = f1_score(
            labels_flat, preds_flat, average="macro", zero_division=0
        )
        try:
            metrics["macro_auc"] = roc_auc_score(
                labels_flat, probs_flat, average="macro"
            )
        except ValueError:
            metrics["macro_auc"] = 0.0

        metrics["micro_precision"] = precision_score(
            labels_flat, preds_flat, average="micro", zero_division=0
        )
        metrics["micro_recall"] = recall_score(
            labels_flat, preds_flat, average="micro", zero_division=0
        )
        metrics["micro_f1"] = f1_score(
            labels_flat, preds_flat, average="micro", zero_division=0
        )
        try:
            metrics["micro_auc"] = roc_auc_score(
                labels_flat, probs_flat, average="micro"
            )
        except ValueError:
            metrics["micro_auc"] = 0.0

c:\Users\almei\Documents\GitHub\research_mecla_objective_paper\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\almei\Documents\GitHub\research_mecla_objective_paper\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\almei\Documents\GitHub\research_mecla_objective_paper\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\almei\Documents\GitHub\research_mecla_objective_paper\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [127]:
metrics

{'best_thresholds': tensor([0.5000, 0.5000, 0.7200, 0.5000, 0.5200, 0.5000, 0.5000, 0.5000, 0.0200,
         0.5000, 0.0200, 0.6300]),
 'best_thresholds_mean': 0.4508333206176758,
 'macro_precision': 0.4166666666666667,
 'macro_recall': 0.4166666666666667,
 'macro_f1': 0.4166666666666667,
 'macro_auc': nan,
 'micro_precision': 1.0,
 'micro_recall': 1.0,
 'micro_f1': 1.0,
 'micro_auc': 1.0}

In [2]:
import jsonlines

file_path = r"experiments\exp_5_mecla_mmbert\metrics_data_split_seed_3.jsonl"

with jsonlines.open(file_path, "r") as reader:
    for obj in reader:
        last_line = obj

In [3]:
with jsonlines.open(file_path, "w") as writer:
    writer.write(last_line)

In [4]:
test_dataset

NameError: name 'test_dataset' is not defined